In [1]:
# 代码作用：读取MES、LIMS数据汇总为A糖煮制(班报) v1
# 2026/01/21 
# 数据治理工程师：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType,DateType
from pyspark.sql.functions import current_timestamp,split,lit,col
from pyspark.sql.functions import col, lit, sum, avg, count, when, round,concat
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import to_date
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# 读取MES数据
mes_sql = """
SELECT 
    T1.season,
    T1.factory,
    T1.team,
    T1.record_code,
    T1.boil_class,
    1 AS discharges,
    1 * 80 AS a_syrup_cube_volume,
    SUM(CASE 
        WHEN T2.material_name LIKE 'A%蜜'
        THEN T2.quantity 
        ELSE 0 
    END) AS a_honey_reboil_quantity  
FROM dwd.dwd_mes_pro_product_record_f_1h T1
LEFT JOIN dwd.dwd_mes_prd_enter_data_f_1h T2
ON T1.record_code = T2.produce_code 
WHERE T1.task_code = 'A0301' AND T1.check_material_type = '甲膏'
GROUP BY 
    T1.season, 
    T1.factory, 
    T1.team, 
    T1.record_code, 
    T1.boil_class, 
    T1.discharges;
"""
mes_df = spark.sql(mes_sql)

# ========== 新增：将放糖罐数 discharges 固定为 1 ==========
mes_df = mes_df.withColumn("discharges", lit(1))

# 读取LIMS样品分析数据（修改：班别转换优化，若原值非编码则保留原值）
lims_sql = """
WITH raw_data AS (
SELECT
sam_sample_original_no,
ord_accept_org_code AS company,
ord_crushing_season AS crushing_season,
ord_accept_date AS record_date,
ord_class,
CASE
WHEN ord_class = '01' THEN '甲班'
WHEN ord_class = '02' THEN '乙班'
WHEN ord_class = '03' THEN '丙班'
ELSE COALESCE(ord_class, '未知班别')
END AS work_shift,
ord_sample_name,
test_name1,
CASE
WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
ELSE NULL
END AS test_value
FROM dwd.dwd_lims_sample_analysis_f_1h
WHERE
ord_proc_st != 'PROC_ZF'
AND ord_accept_org_code = 'FNR'
AND ord_productioncategory = '榨蔗生产'
AND (ord_sample_name LIKE '%甲膏%' OR ord_sample_name LIKE '%原糖%' OR ord_sample_name LIKE '%甲原蜜%')
AND test_name1 IN ('锤度', '视纯度', '色值')
),
paste_a_data AS (
SELECT
sam_sample_original_no,
company,
crushing_season,
record_date AS paste_record_date,
work_shift AS paste_work_shift,
MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS paste_a_brix,
MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS paste_a_apparent_purity
FROM raw_data
WHERE ord_sample_name LIKE '%甲膏%'
GROUP BY sam_sample_original_no, company, crushing_season, record_date, work_shift
),
raw_sugar_data AS (
SELECT
sam_sample_original_no,
crushing_season,
MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS raw_sugar_color_value
FROM raw_data
WHERE ord_sample_name LIKE '%原糖%'
GROUP BY sam_sample_original_no, crushing_season
),
honey_a_data AS (
SELECT
sam_sample_original_no,
crushing_season,
MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS honey_a_apparent_purity
FROM raw_data
WHERE ord_sample_name LIKE '%甲原蜜%'
GROUP BY sam_sample_original_no, crushing_season
)
SELECT
p.sam_sample_original_no,
p.company,
p.crushing_season,
p.paste_record_date,
p.paste_work_shift,
p.paste_a_brix,
p.paste_a_apparent_purity,
r.raw_sugar_color_value,
h.honey_a_apparent_purity
FROM paste_a_data p
LEFT JOIN raw_sugar_data r ON p.sam_sample_original_no = r.sam_sample_original_no AND p.crushing_season = r.crushing_season
LEFT JOIN honey_a_data h ON p.sam_sample_original_no = h.sam_sample_original_no AND p.crushing_season = h.crushing_season;
"""
lims_df = spark.sql(lims_sql)

# 读取lims班报糖浆数据（班别转换优化，与上述逻辑一致）
lims_ctj_sql = """
SELECT 
  season,
  gongsidm,
  rq,
  CASE 
    WHEN banbiedm = '01' THEN '甲班'
    WHEN banbiedm = '02' THEN '乙班'
    WHEN banbiedm = '03' THEN '丙班'
    ELSE COALESCE(banbiedm, '未知班别')
  END AS banbie_name,
  banbiedm,
  bx
FROM dwd.dwd_lims_sugar_cane_juice_syrup_batch_report_f_1h 
WHERE leibiedm = '粗糖浆' 
  AND gongsidm = 'FNR';
"""
lims_ctj_df = spark.sql(lims_ctj_sql)
lims_ctj_df = lims_ctj_df.withColumnRenamed("season", "ctj_season")  

# 读取lims班报实绩数据（班别转换优化，与上述逻辑一致）
lims_bbsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
    WHEN tb_banbiedm = '01' THEN '甲班'
    WHEN tb_banbiedm = '02' THEN '乙班'
    WHEN tb_banbiedm = '03' THEN '丙班'
    ELSE COALESCE(tb_banbiedm, '未知班别')
END AS banbie_name,
sj_zhazheliang
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)
lims_bbsj_df = lims_bbsj_df.withColumnRenamed("season", "bbsj_season")   


# ========== 核心：构建明细数据 ==========
mes_df = mes_df.withColumn("connect_code", split(mes_df.record_code, "-")[1])

# 内连接mes_df和lims_df（连接条件：connect_code = sam_sample_original_no）
mes_lims_df_v1 = mes_df.join(
    lims_df,
    mes_df.connect_code == lims_df.sam_sample_original_no,
    how="inner"
)

# ========== 新增：将team字段替换为LIMS的班别（paste_work_shift） ==========
mes_lims_df_v1 = mes_lims_df_v1.withColumn("team", col("paste_work_shift"))

# 选择需要的列（此时team已为LIMS班别）
list1 = ["season","factory","paste_record_date","team","record_code","discharges","a_syrup_cube_volume",
         "a_honey_reboil_quantity","paste_a_brix","paste_a_apparent_purity","raw_sugar_color_value","honey_a_apparent_purity"]
mes_lims_df_v1 = mes_lims_df_v1.select(*list1)

# ========== 核心：添加7个空值列（非链式调用，无缩进风险） ==========  
# 1. 实绩榨蔗量
mes_lims_df_v2 = mes_lims_df_v1.withColumn("actual_cane_crushing_quantity", lit(None))
# 2. 粗糖浆锤度
mes_lims_df_v2 = mes_lims_df_v2.withColumn("raw_syrup_brix", lit(None))
# 3. A糖膏固溶物对蔗比
mes_lims_df_v2 = mes_lims_df_v2.withColumn("a_syrup_cube_volume_to_cane_ratio", lit(None))
# 4. A糖膏蜜纯度差（核心：两列相减计算）
mes_lims_df_v2 = mes_lims_df_v2.withColumn("a_honey_purity_difference", col("paste_a_apparent_purity") - col("honey_a_apparent_purity"))
# 5. A糖膏锤度合格率
mes_lims_df_v2 = mes_lims_df_v2.withColumn("a_paste_brix_qualified_rate", lit(None))
# 6. A糖膏纯度合格率
mes_lims_df_v2 = mes_lims_df_v2.withColumn("a_paste_purity_qualified_rate", lit(None))
# 7. 原糖色值合格率
mes_lims_df_v2 = mes_lims_df_v2.withColumn("raw_sugar_color_qualified_rate", lit(None))
# 8. 粗糖浆量
mes_lims_df_v2 = mes_lims_df_v2.withColumn("raw_syrup_quantity", lit(None))
# 9. A 糖膏固溶物量
mes_lims_df_v2 = mes_lims_df_v2.withColumn("a_paste_solid_solute_quantity", lit(None))

list2 = ["season","factory","paste_record_date","team","record_code","actual_cane_crushing_quantity","raw_syrup_quantity",
         "raw_syrup_brix","a_honey_reboil_quantity","honey_a_apparent_purity","discharges","a_syrup_cube_volume",
        "a_syrup_cube_volume_to_cane_ratio","paste_a_brix","a_paste_solid_solute_quantity","paste_a_apparent_purity"
         ,"raw_sugar_color_value","a_honey_purity_difference","a_paste_brix_qualified_rate","a_paste_purity_qualified_rate"
        ,"raw_sugar_color_qualified_rate"]

col_name_mapping = {
    "season": "榨季",
    "factory": "工厂",
    "paste_record_date": "日期",
    "team": "班别",
    "record_code": "编号",
    "actual_cane_crushing_quantity": "榨蔗量",
    "raw_syrup_quantity": "粗糖浆量",
    "raw_syrup_brix": "粗糖浆锤度",
    "a_honey_reboil_quantity": "A蜜回煮量m3",
    "honey_a_apparent_purity": "A蜜纯度AP",
    "discharges": "放糖罐数",
    "a_syrup_cube_volume": "放糖立方数",
    "a_syrup_cube_volume_to_cane_ratio": "A糖膏立方数对蔗比",
    "paste_a_brix": "A糖膏锤度93～97",
    "a_paste_solid_solute_quantity": "A糖膏固溶物量",
    "paste_a_apparent_purity": "A糖膏纯度82～88",
    "raw_sugar_color_value": "原糖色值IU≤800IU",
    "a_honey_purity_difference": "A糖膏蜜纯度差",
    "a_paste_brix_qualified_rate": "A膏锤度合格率",
    "a_paste_purity_qualified_rate": "A膏纯度合格率",
    "raw_sugar_color_qualified_rate": "原糖色值合格率",
    "source_flg":"来源系统",
    "dw_update_time":"ETL跑批时间"
}


mes_lims_df_v2 = mes_lims_df_v2.select(*list2)


# ========== 核心：分组求小计（严格按照list2字段顺序） ==========
mes_lims_subtotal = mes_lims_df_v2 \
    .groupBy("season", "factory", "paste_record_date", "team")\
    .agg(
        round(lit(0), 0).alias("actual_cane_crushing_quantity"),  # 榨蔗量：设为0
        lit(None).alias("raw_syrup_quantity"),  # 粗糖浆量：设为空值
        round(lit(0), 0).alias("raw_syrup_brix"),# 粗糖浆锤度：设为0
        round(sum("a_honey_reboil_quantity"), 0).alias("a_honey_reboil_quantity"),  # A蜜回煮：求和
        round(avg("honey_a_apparent_purity"), 3).alias("honey_a_apparent_purity"),  # A蜜纯度：均值
        round(sum("discharges"), 0).alias("discharges"),  # 放糖罐数：求和（每个记录为1，求和即为记录数）
        round(sum("a_syrup_cube_volume"), 0).alias("a_syrup_cube_volume"), # A糖放糖立方数：求和
        round(lit(0), 0).alias("a_syrup_cube_volume_to_cane_ratio"), # A糖膏立方数对蔗比：设为0
        round(avg("paste_a_brix"), 2).alias("paste_a_brix"),  # A膏锤度：均值
        round(
            sum("a_syrup_cube_volume") * 1.51 * avg("paste_a_brix") / 100,
            3
        ).alias("a_paste_solid_solute_quantity"),  # A糖膏固溶物量：公式计算
        round(avg("paste_a_apparent_purity"), 3).alias("paste_a_apparent_purity"),  # A膏纯度：均值
        round(avg("raw_sugar_color_value"), 3).alias("raw_sugar_color_value"),  # 原糖色值IU：均值
        round(
            avg("paste_a_apparent_purity") - avg("honey_a_apparent_purity"),
            3
        ).alias("a_honey_purity_difference"),  # A糖膏蜜纯度差：均值相减
        round(
            count(when((col("paste_a_brix") >= 93) & (col("paste_a_brix") <= 97), 1)) / count("paste_a_brix"),
            3
        ).alias("a_paste_brix_qualified_rate"),  # A膏锤度合格率
        round(
            count(when((col("paste_a_apparent_purity") >= 82) & (col("paste_a_apparent_purity") <= 88), 1)) / count("paste_a_apparent_purity"),
            3
        ).alias("a_paste_purity_qualified_rate"),  # A膏纯度合格率
        round(
            count(when(col("raw_sugar_color_value") <= 800, 1)) / count("raw_sugar_color_value"),
            3
        ).alias("raw_sugar_color_qualified_rate"),  # 原糖色值合格率
    )
    

# 获取榨蔗量（关联条件使用优化后的班别）
mes_lims_subtotal = mes_lims_subtotal.join(
    lims_bbsj_df,
    (mes_lims_subtotal.factory == lims_bbsj_df.tb_gongsidm) &
    (mes_lims_subtotal.paste_record_date == lims_bbsj_df.tb_rq) &
    (mes_lims_subtotal.team == lims_bbsj_df.banbie_name),   # team此时为LIMS班别
    how="left"
)   
mes_lims_subtotal = mes_lims_subtotal \
    .withColumn(
        "actual_cane_crushing_quantity",
        round(col("sj_zhazheliang"), 3)
    ) \
    .drop(*lims_bbsj_df.columns)


# 获取粗糖浆锤度（关联条件使用优化后的班别）
mes_lims_subtotal = mes_lims_subtotal.join(
    lims_ctj_df,
    (mes_lims_subtotal.factory == lims_ctj_df.gongsidm) &
    (mes_lims_subtotal.paste_record_date == lims_ctj_df.rq) &
    (mes_lims_subtotal.team == lims_ctj_df.banbie_name),   # team此时为LIMS班别
    how="left"
)   
mes_lims_subtotal = mes_lims_subtotal \
    .withColumn(
        "raw_syrup_brix",
        round(col("bx"), 3)
    ) \
    .drop(*lims_ctj_df.columns)


# 计算A糖膏立方数对蔗比
mes_lims_subtotal = mes_lims_subtotal \
    .withColumn(
        "a_syrup_cube_volume_to_cane_ratio",
        when(
            (col("actual_cane_crushing_quantity") != 0) & (col("actual_cane_crushing_quantity").isNotNull()),
            round(col("a_syrup_cube_volume") / col("actual_cane_crushing_quantity"), 3)
        ).otherwise(None)
    )

# 改变班次保证与明细不一样
mes_lims_subtotal = mes_lims_subtotal \
    .withColumn(
        "team",
        concat(col("team"), lit("-小计"))
    )

mes_lims_subtotal = mes_lims_subtotal.withColumn("record_code", lit(None))
mes_lims_subtotal = mes_lims_subtotal.select(*list2)


# ========== 核心：计算合计数据（分组：season、factory、paste_record_date） ==========
mes_lims_total = mes_lims_subtotal \
    .groupBy("season", "factory", "paste_record_date") \
    .agg(
        round(sum("actual_cane_crushing_quantity"), 4).alias("actual_cane_crushing_quantity"),
        lit(None).alias("raw_syrup_quantity"),
        round(avg("raw_syrup_brix"), 3).alias("raw_syrup_brix"),
        round(sum("a_honey_reboil_quantity"), 0).alias("a_honey_reboil_quantity"),
        round(avg("honey_a_apparent_purity"), 3).alias("honey_a_apparent_purity"),
        round(sum("discharges"), 0).alias("discharges"),
        round(sum("a_syrup_cube_volume"), 0).alias("a_syrup_cube_volume"),
        when(
            (sum("actual_cane_crushing_quantity") != 0) & (sum("actual_cane_crushing_quantity").isNotNull()),
            round(sum("a_syrup_cube_volume") / sum("actual_cane_crushing_quantity"), 3)
        ).otherwise(None).alias("a_syrup_cube_volume_to_cane_ratio"),
        round(avg("paste_a_brix"), 3).alias("paste_a_brix"),
        round(sum("a_paste_solid_solute_quantity"), 3).alias("a_paste_solid_solute_quantity"),
        round(avg("paste_a_apparent_purity"), 3).alias("paste_a_apparent_purity"),
        round(avg("raw_sugar_color_value"), 3).alias("raw_sugar_color_value"),
        round(avg("a_honey_purity_difference"), 3).alias("a_honey_purity_difference"),
        round(avg("a_paste_brix_qualified_rate"), 3).alias("a_paste_brix_qualified_rate"),
        round(avg("a_paste_purity_qualified_rate"), 3).alias("a_paste_purity_qualified_rate"),
        round(avg("raw_sugar_color_qualified_rate"), 3).alias("raw_sugar_color_qualified_rate")
    ) \
    .withColumn("team", lit("合计")) \
    .withColumn("record_code", lit(None))

mes_lims_total = mes_lims_total.select(*list2)

mes_lims_complete =  mes_lims_df_v2.unionByName(mes_lims_subtotal, allowMissingColumns=True)
mes_lims_complete = mes_lims_complete.unionByName(mes_lims_total, allowMissingColumns=True)
mes_lims_complete = mes_lims_complete \
    .withColumn("source_flg", lit("MES+LIMS")) \
    .withColumn("dw_update_time", current_timestamp())


# 配置信息
english_table_name = "app.app_mes_a_syrup_continuous_cook_control_params_team_stats_f_1h"
chinese_table_comment = "A糖连续罐煮制控制参数统计（班统计）"

# Spark类型转Hive类型
def spark_type_to_hive_type(spark_type):
    if isinstance(spark_type, StringType):
        return "string"
    elif isinstance(spark_type, DecimalType):
        return f"decimal({spark_type.precision}, {spark_type.scale})"
    elif isinstance(spark_type, DateType):
        return "date"
    elif isinstance(spark_type, TimestampType):
        return "timestamp"
    else:
        return "string"

# 字符串字段处理
char_columns = ["season", "factory", "team", "record_code", "source_flg"]
for col_name in char_columns:
    if col_name in mes_lims_complete.columns:
        mes_lims_complete = mes_lims_complete.withColumn(col_name, col(col_name).cast(StringType()))

# 时间字段处理
if "paste_record_date" in mes_lims_complete.columns:
    mes_lims_complete = mes_lims_complete.withColumn("paste_record_date", to_date(col("paste_record_date")).cast(DateType()))

if "dw_update_time" in mes_lims_complete.columns:
    mes_lims_complete = mes_lims_complete.withColumn("dw_update_time", col("dw_update_time").cast(TimestampType()))

# 数值字段处理
exclude_columns = char_columns + ["paste_record_date", "dw_update_time"]
for col_name in mes_lims_complete.columns:
    if col_name not in exclude_columns:
        mes_lims_complete = mes_lims_complete.withColumn(col_name, col(col_name).cast(DecimalType(16, 3)))

# 写入表
print('写入数据')
mes_lims_complete \
    .write \
    .mode("overwrite") \
    .format("orc") \
    .saveAsTable(english_table_name)

# 批量添加字段注释
alter_sqls = []
for field_name in mes_lims_complete.columns:
    field_comment = col_name_mapping.get(field_name, field_name).replace("'", "''")
    hive_type = spark_type_to_hive_type(mes_lims_complete.schema[field_name].dataType)
    alter_sql = f"ALTER TABLE {english_table_name} CHANGE COLUMN `{field_name}` `{field_name}` {hive_type} COMMENT '{field_comment}'"
    alter_sqls.append(alter_sql)

for sql in alter_sqls:
    spark.sql(sql)

# 添加表注释
processed_table_comment = chinese_table_comment.replace("'", "''")
spark.sql(f"ALTER TABLE {english_table_name} SET TBLPROPERTIES ('comment'='{processed_table_comment}')")

print("注释添加完成")

Spark 启动中...
✅ 数据已成功写入Hive表：app.app_mes_brown_sugar_cook_control_stats_f_1h
📝 表注释：金砂糖煮糖控制统计（班报），包含色值、干燥失重、粒径、接单日期等关键指标
🕒 写入时间：2026-03-11 16:40:17
